# Exercise 03 — GeoJSON + HTML
### 🗺️ Putting sensors on a map

Sensor data only makes sense **in context** — where is the sensor?
**GeoJSON** is the standard format for geographic data built on top of JSON.

In this exercise you will:
- Build GeoJSON objects from scratch in Python
- Attach sensor readings as feature properties
- Generate an HTML file that renders them on an interactive map

---

## 1️⃣  GeoJSON anatomy

A GeoJSON `Feature` wraps a geometry and a `properties` dict — the perfect container for a sensor.

In [ ]:
import json

# A single sensor as a GeoJSON Feature
feature = {
    "type": "Feature",
    "geometry": {
        "type": "Point",
        "coordinates": [7.6869, 45.0703]   # [longitude, latitude]  ← note the order!
    },
    "properties": {
        "sensor_id": "TRN-001",
        "location": "Piazza Castello",
        "temperature_c": 22.4,
        "pm25_ugm3": 12.1
    }
}

print(json.dumps(feature, indent=2))

## 2️⃣  Build a FeatureCollection for the whole network

A `FeatureCollection` groups many features — one per sensor.

In [ ]:
# Raw sensor data with coordinates
sensors = [
    {"id": "TRN-001", "name": "Piazza Castello",  "lon": 7.6869,  "lat": 45.0703, "temp": 22.4, "pm25": 12.1, "humidity": 58},
    {"id": "TRN-002", "name": "Piazza Vittorio",  "lon": 7.6943,  "lat": 45.0647, "temp": 23.1, "pm25": 18.4, "humidity": 61},
    {"id": "TRN-003", "name": "Mirafiori",        "lon": 7.6332,  "lat": 44.9980, "temp": 21.0, "pm25": 35.2, "humidity": 70},
    {"id": "TRN-004", "name": "Lingotto",         "lon": 7.6757,  "lat": 45.0333, "temp": 24.5, "pm25": 22.7, "humidity": 52},
    {"id": "TRN-005", "name": "Parco del Po",     "lon": 7.7100,  "lat": 45.0550, "temp": 20.8, "pm25":  8.3, "humidity": 75},
]

def sensor_to_feature(s):
    """Convert a sensor dict to a GeoJSON Feature."""
    return {
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [s["lon"], s["lat"]]
        },
        "properties": {
            "sensor_id":    s["id"],
            "location":     s["name"],
            "temperature_c": s["temp"],
            "humidity_pct": s["humidity"],
            "pm25_ugm3":    s["pm25"],
            "alert":        s["pm25"] > 20   # boolean flag for high pollution
        }
    }

geojson = {
    "type": "FeatureCollection",
    "features": [sensor_to_feature(s) for s in sensors]
}

print(f"FeatureCollection with {len(geojson['features'])} features")
print(json.dumps(geojson, indent=2))

## 3️⃣  Save as a .geojson file

You can open this file directly in the **GeoJSON Viewer** HTML tool from the previous session!

In [ ]:
with open("turin_sensors.geojson", "w") as f:
    json.dump(geojson, f, indent=2)

print("Saved to turin_sensors.geojson ✓")
print("You can now load this file into the GeoJSON Viewer HTML tool!")

## 4️⃣  Generate a standalone HTML map

Now we go further: Python **writes** an HTML file that includes the GeoJSON data and renders it with Leaflet.
This is exactly what IoT dashboards do — the server generates the page dynamically.

In [ ]:
geojson_str = json.dumps(geojson)

html = f"""
<!DOCTYPE html>
<html>
<head>
  <meta charset="UTF-8">
  <title>SmartCity Turin — Sensor Map</title>
  <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
  <style>
    body {{ margin: 0; font-family: sans-serif; }}
    #map {{ height: 100vh; }}
    .popup {{ font-size: 13px; line-height: 1.6; }}
    .alert {{ color: #c0392b; font-weight: bold; }}
  </style>
</head>
<body>
<div id="map"></div>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script>
  var map = L.map('map').setView([45.05, 7.67], 12);

  L.tileLayer('https://{{s}}.basemaps.cartocdn.com/light_all/{{z}}/{{x}}/{{y}}{{r}}.png', {{
    attribution: '&copy; OpenStreetMap &copy; CARTO'
  }}).addTo(map);

  var data = {geojson_str};

  L.geoJSON(data, {{
    pointToLayer: function(feature, latlng) {{
      var pm25 = feature.properties.pm25_ugm3;
      var color = pm25 > 20 ? '#e74c3c' : '#2ecc71';  // red if polluted
      return L.circleMarker(latlng, {{
        radius: 10,
        fillColor: color,
        color: '#fff',
        weight: 2,
        fillOpacity: 0.85
      }});
    }},
    onEachFeature: function(feature, layer) {{
      var p = feature.properties;
      var alertHtml = p.alert ? '<br><span class="alert">⚠️ HIGH PM2.5</span>' : '';
      layer.bindPopup(
        '<div class="popup">'
        + '<b>' + p.location + '</b> (' + p.sensor_id + ')'
        + '<br>🌡️ Temp: ' + p.temperature_c + ' °C'
        + '<br>💧 Humidity: ' + p.humidity_pct + '%'
        + '<br>🌫️ PM2.5: ' + p.pm25_ugm3 + ' µg/m³'
        + alertHtml
        + '</div>'
      );
    }}
  }}).addTo(map);
</script>
</body>
</html>
"""

with open("sensor_map.html", "w") as f:
    f.write(html)

print("Map generated → open sensor_map.html in your browser!")
print("Green dots = clean air, Red dots = PM2.5 > 20 µg/m³")

## 5️⃣  Add a pollution zone (Polygon)

GeoJSON supports more than just points. Let's add a polygon to mark a high-traffic industrial zone.

In [ ]:
# A simple polygon around the Mirafiori industrial area
industrial_zone = {
    "type": "Feature",
    "geometry": {
        "type": "Polygon",
        "coordinates": [[
            [7.620, 44.990],
            [7.650, 44.990],
            [7.650, 45.005],
            [7.620, 45.005],
            [7.620, 44.990]   # close the ring — first == last
        ]]
    },
    "properties": {
        "name": "Mirafiori Industrial Zone",
        "type": "industrial"
    }
}

geojson_with_zone = {
    "type": "FeatureCollection",
    "features": geojson["features"] + [industrial_zone]
}

print(f"Total features now: {len(geojson_with_zone['features'])}")
print("(5 points + 1 polygon)")

with open("turin_sensors_zones.geojson", "w") as f:
    json.dump(geojson_with_zone, f, indent=2)

print("Saved to turin_sensors_zones.geojson — load it in the GeoJSON Viewer!")

---
## 🏆 Challenges

1. Add a `LineString` feature connecting the 5 sensors in order (a patrol route).
2. In the HTML map, make the circle **radius proportional to PM2.5** level (e.g. `radius = pm25 / 3`).
3. Add a second HTML map that colours sensors by **temperature** instead of pollution.
4. Write a function `bbox(geojson)` that returns the bounding box `[min_lon, min_lat, max_lon, max_lat]` of all features.

In [ ]:
# Your code here
